# 06 · POS demo: what the cashier sees

A working till prototype on top of the engine. Pick a branch and a time, ring the first item, and the screen shows
the suggestion (or stays quiet), the price, and a staff-only line explaining why. **Accept / Decline** buttons log
the response: this log is the dataset Stories does not have today, and the pilot (notebook 05) runs on it.

- **No Python at all?** Open [`app/index.html`](../app/index.html) in any browser: the same engine as a full till app,
  with the learning engine (notebook 07) and a business-case tab.
- The first cells render example cards directly in the notebook (no extra install).
- The last cell launches an interactive app with **Gradio** in Google Colab (`share=True` prints a public link for
  the presentation). Locally, `pip install gradio` first.

In [1]:
# Setup: works in Google Colab, local Jupyter, and from the repo root
import os, sys, subprocess
REPO_URL = "https://github.com/mouhammed-soumaille/EECE-490-Hackathon.git"
if "google.colab" in sys.modules:
    if not os.path.exists("/content/EECE-490-Hackathon"):
        subprocess.run(["git", "clone", "-q", REPO_URL, "/content/EECE-490-Hackathon"], check=True)
    os.chdir("/content/EECE-490-Hackathon")
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.path.join(os.getcwd(), "src"))

import warnings; warnings.filterwarnings("ignore")
import html, datetime as dt
import pandas as pd
from storiespair import data, models, engine as E

lines = data.load_lines()
orders = data.build_orders(lines)
added = data.other_side_first_item(lines, orders)
ue = data.unit_economics(lines)
cfg = E.load_config()
pm = models.PairModels.fit(orders, lines, added)
engine = E.PairEngine(pm, ue, cfg["compatibility"], E.Assumptions.from_config(cfg), E.Guardrails.from_config(cfg))
MENU = {side: engine.items[side] for side in ["drink", "food"]}
print(f"Engine ready: {len(MENU['drink'])} drinks, {len(MENU['food'])} food items, 3 branches.")

Engine ready: 23 drinks, 22 food items, 3 branches.


In [2]:
# Inline styles so the cards also render on GitHub, which strips <style> blocks
ST = {
    "card": "font-family:Inter,system-ui,sans-serif;max-width:560px;border:1px solid #e3e1db;border-radius:14px;"
            "background:#fcfcfb;color:#0b0b0b;overflow:hidden;margin:8px 0",
    "top": "padding:12px 18px;border-bottom:1px solid #eeece6;font-size:13px;color:#52514e",
    "body": "padding:16px 18px",
    "name": "font-size:12px;letter-spacing:.06em;text-transform:uppercase;color:#2a78d6;font-weight:700",
    "line": "font-size:20px;font-weight:700;margin:6px 0 2px",
    "price": "font-size:15px;color:#52514e",
    "say": "margin-top:12px;padding:10px 12px;background:#eef4fc;border-radius:10px;font-size:14px",
    "staff": "padding:12px 18px;background:#f4f3ef;font-size:12px;color:#52514e;line-height:1.45",
    "quiet": "font-size:15px;color:#52514e",
}


class Card:
    """Renders a suggestion as the cashier's screen (works in Jupyter, Colab and on GitHub)."""

    def __init__(self, branch, when, basket, r):
        self.branch, self.when, self.basket, self.r = branch, pd.Timestamp(when), basket, r

    def _repr_html_(self):
        r = self.r
        top = (f"<div style='{ST['top']}'>Stories {html.escape(self.branch)} · {self.when:%a %H:%M}"
               f" &nbsp;|&nbsp; Basket: {html.escape(', '.join(self.basket))}</div>")
        if not r["offer"]:
            body = f"<div style='{ST['body']}'><div style='{ST['quiet']}'>No suggestion. {html.escape(r['reason'])}</div></div>"
            return f"<div style='{ST['card']}'>{top}{body}</div>"
        saving = f" · save ${r['saving']:.2f}" if r["saving"] > 0 else ""
        item = r["suggested_item"].lower()
        article = "an" if item[0] in "aeiou" else "a"
        say = (f"“Would you like to make it a {r['pairing_name']}? Your {r['first_item'].lower()} with {article} "
               f"{item}, ${r['combo_total']:.2f} together.”")
        body = (f"<div style='{ST['body']}'><div style='{ST['name']}'>{r['pairing_name']}</div>"
                f"<div style='{ST['line']}'>{html.escape(r['first_item'])} + {html.escape(r['suggested_item'])}</div>"
                f"<div style='{ST['price']}'>${r['combo_total']:.2f} total · add-on ${r['item_price']:.2f}{saving}</div>"
                f"<div style='{ST['say']}'>{html.escape(say)}</div></div>")
        staff = (f"<div style='{ST['staff']}'><b>Staff only</b> · expected gain ${r['expected_gain']:.2f} per suggestion · "
                 f"{r['Q']:.0%} of these orders pair without help<br>{html.escape(r['reason'])}</div>")
        return f"<div style='{ST['card']}'>{top}{body}{staff}</div>"

    def __repr__(self):
        r = self.r
        return (f"{self.branch} {self.when:%H:%M} {self.basket} -> "
                + (f"{r['suggested_item']} (${r['combo_total']:.2f} total)" if r["offer"] else "no suggestion"))


def till(branch, when, basket, declined=None):
    return Card(branch, when, basket, engine.recommend(branch, when, basket, declined=declined))

## Example screens

In [3]:
display(till("Harbour", "2026-06-03 08:40", ["Cappuccino"]))
display(till("Garden", "2026-06-03 21:15", ["Iced Matcha"]))
display(till("Uptown", "2026-06-03 13:05", ["Chicken Pesto Sandwich"]))

Harbour 08:40 ['Cappuccino'] -> Egg Sandwich ($12.81 total)

Garden 21:15 ['Iced Matcha'] -> Butter Croissant ($10.50 total)

Uptown 13:05 ['Chicken Pesto Sandwich'] -> Peach Iced Tea ($14.59 total)

Same drink, different branch: Garden's higher unit costs change the best suggestion.

In [4]:
display(till("Harbour", "2026-06-03 13:05", ["Iced Latte"]))
display(till("Garden", "2026-06-03 13:05", ["Iced Latte"]))

Harbour 13:05 ['Iced Latte'] -> Chicken Pesto Sandwich ($16.91 total)

Garden 13:05 ['Iced Latte'] -> Egg Sandwich ($13.50 total)

The customer declines, so the engine never repeats that offer and moves to the next best one; a complete pair gets
no suggestion at all.

In [5]:
display(till("Harbour", "2026-06-03 08:40", ["Cappuccino"], declined=["Egg Sandwich"]))
display(till("Harbour", "2026-06-03 08:40", ["Cappuccino", "Butter Croissant"]))

Harbour 08:40 ['Cappuccino'] -> Avocado Toast ($14.17 total)

Harbour 08:40 ['Cappuccino', 'Butter Croissant'] -> no suggestion

## Interactive app (Google Colab)

In [6]:
import importlib.util
if importlib.util.find_spec("gradio") is None and "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gradio"], check=True)

if importlib.util.find_spec("gradio") is None:
    print("Gradio is not installed here. Run this notebook in Google Colab (it installs automatically) "
          "or `pip install gradio` locally, then re-run this cell.")
else:
    import gradio as gr

    LOG_COLS = ["time", "branch", "first item", "suggested", "price", "response"]

    def ring(branch, hour, first_item, declined):
        when = pd.Timestamp("2026-06-03") + pd.Timedelta(hours=int(hour), minutes=5)
        r = engine.recommend(branch, when, [first_item], declined=declined)
        return Card(branch, when, [first_item], r)._repr_html_(), r

    def on_ring(branch, hour, first_item):
        card_html, r = ring(branch, hour, first_item, [])
        return card_html, r, []

    def on_respond(response, branch, hour, first_item, r, declined, log):
        if not r or not r.get("offer"):
            return gr.update(), r, declined, log
        log = log if isinstance(log, pd.DataFrame) else pd.DataFrame(log or [], columns=LOG_COLS)
        log = log[(log.astype(str) != "").any(axis=1)] if len(log) else log  # drop the blank starter row
        row = {"time": dt.datetime.now().strftime("%H:%M:%S"), "branch": branch, "first item": first_item,
               "suggested": r["suggested_item"], "price": r["item_price"], "response": response}
        log = pd.concat([log, pd.DataFrame([row])], ignore_index=True)
        if response == "declined":
            declined = declined + [r["suggested_item"]]
            card_html, r = ring(branch, hour, first_item, declined)
            return card_html, r, declined, log
        done = Card(branch, pd.Timestamp("2026-06-03") + pd.Timedelta(hours=int(hour)), [first_item, r["suggested_item"]],
                    {"offer": False, "reason": "Pair complete. Accepted and logged."})
        return done._repr_html_(), r, declined, log

    with gr.Blocks(title="Stories Pair · till demo") as demo:
        gr.Markdown("## Stories Pair · till demo\nRing the first item. The engine suggests the other half of the pair.")
        with gr.Row():
            branch = gr.Dropdown(["Harbour", "Garden", "Uptown"], value="Harbour", label="Branch")
            hour = gr.Slider(7, 23, value=9, step=1, label="Hour")
            first = gr.Dropdown(MENU["drink"] + MENU["food"], value="Cappuccino", label="First item rung")
        ring_btn = gr.Button("Ring it", variant="primary")
        card = gr.HTML()
        with gr.Row():
            acc = gr.Button("Customer accepted")
            dec = gr.Button("Customer declined")
        log = gr.Dataframe(value=pd.DataFrame(columns=LOG_COLS), headers=LOG_COLS,
                           label="Response log (the data the pilot collects)")
        state_r, state_declined = gr.State(None), gr.State([])
        ring_btn.click(on_ring, [branch, hour, first], [card, state_r, state_declined])
        acc.click(lambda *a: on_respond("accepted", *a), [branch, hour, first, state_r, state_declined, log],
                  [card, state_r, state_declined, log])
        dec.click(lambda *a: on_respond("declined", *a), [branch, hour, first, state_r, state_declined, log],
                  [card, state_r, state_declined, log])

    demo.launch(share="google.colab" in sys.modules, debug=False)

Gradio is not installed here. Run this notebook in Google Colab (it installs automatically) or `pip install gradio` locally, then re-run this cell.
